# 🌙 رفيق — التشغيل داخل Google Colab (نماذجك الأصلية حرفيًا)

تشغيل المساعد الصوتي العربي بالكومة الكاملة: **Jais-ASR** (استماع) + **Jais-3-7B** (تفكير) + **SILMA TTS** (كلام)، مع خادم FastAPI وواجهة عربية كاملة، ورابط عمومي يتوافق مع الميكروفون (WebSocket).

## ⚠️ الخطوة صفر — اختر كارت الشاشة أولًا
كولاب ← **بيئة التشغيل / Runtime** ← **تغيير نوع بيئة التشغيل / Change runtime type** ← **GPU**، ثم افعل الخلايا بالترتيب.

| كارت الشاشة | الذاكرة | التقدير |
|---|---|---|
| **A100 40GB** (Colab Pro) | كافية بفاضل | 🟢 النماذج الثلاثة كاملة على الكارت — رد صوتي 3–6 ثوان |
| **L4 24GB** (Colab Pro) | ضيّقة قليلًا | 🟡 يعمل مع ترحيل جزئي بسيط إلى الذاكرة |
| **T4 16GB** (المجاني) | غير كافية | 🔴 سيُترحَّل جزء كبير للقرص تلقائيًا — تجربة الصوت بطيئة (النص و API سريعان) |

> 💡 النماذج الثلاثة معًا تحتاج **~27.5GB** على كارت الشاشة. الكود يحتوي مسبقًا `AUTO_OFFLOAD_ON_OOM` فيترحّل تلقائيًا عند نقص الذاكرة فلن ينهار النظام، لكن A100 أسرع كثيرًا.

## 🚀 خطوات التشغيل (بالترتيب)
1. ارفع **rafeeq-main.zip** (الموجود عندك على سطح المكتب) عبر الشرائح «الملفات / Files» على يسار الكولاب — أو صعّده من الخلية 2 (ستظهر نافذة رفع عند التجاوب).
2. نفّذ الخلايا بالترتيب: فحص ← المشروع ← المكتبات ← النماذج ← الخادم ← النفق العمومي.
3. افتح الرابط العمومي الذي ستطبعه الخلية 7 ← أنشئ **حساب المالك** (يُغلق للأبد بعد أول إنشاء).
4. (اختياري) خلية 5 تحفظ النماذج في Drive لتكون الجلسات القادمة أسرع بلا تنزيل.

> 📝 النماذج تُنزَّل **رسميًا** من HuggingFace بنفس أسماء ملفاتك حرفيًا: `IFM/Jais-ASR`, `inception42/Jais-3-7B`, `silma-ai/silma-tts` — لا ترفع المجلدات الكبيرة يدويًا.
>
> 🚫 ملاحظة مهمة: كولاب للتجربة والعرض — يقطع الجلسة بعد 12 ساعة أو مع الخمول، ولا يُنصح به لتشغيل الفوترة لعملاء حقيقيين. للإنتاج استخدم RunPod أو Modal كما في `deploy/README.md` بنفس الخطوات.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "❌ كارت الشاشة غير مفعّل — بيئة التشغيل ← تغيير نوع بيئة التشغيل ← GPU ثم أعد تشغيل الجلسة"
)

_gpu = torch.cuda.get_device_name(0)
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {_gpu} | الذاكرة: {_vram:.0f} GB")
print(f"   PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if "A100" in _gpu:
    print("🟢 A100 — الجودة الأمثل: النماذج الثلاثة كاملة على الكارت.")
elif "L4" in _gpu or "A40" in _gpu:
    print("🟡 L4/A40 — يعمل مع ترحيل جزئي بسيط.")
else:
    print("🔴 كارت أصغر من المطلوب: الاستماع والتفكير سيعملان مع ترحيل للقرص — الصوت سيكون بطيئًا. الأفضل A100 من Colab Pro.")


In [ ]:
import glob
import os
import subprocess
import zipfile
from pathlib import Path

PROJ = Path("/content/rafeeq-main")

if (PROJ / "app" / "main.py").is_file():
    print("✅ المشروع موجود مسبقًا:", PROJ)
else:
    candidates = [
        z
        for z in (
            glob.glob("/content/*.zip")
            or glob.glob("/content/drive/MyDrive/**/rafeeq*.zip", recursive=True)
        )
        if "rafeeq" in os.path.basename(z).lower()
    ]
    if candidates:
        zsrc = candidates[0]
        print("📂 فك الضغط:", zsrc)
        with zipfile.ZipFile(zsrc) as zf:
            zf.extractall("/content")
        # لو كان الزيب يحتوي مجلدًا باسم مختلف — إعادة التسمية تلقائيًا
        if not (PROJ / "app" / "main.py").is_file():
            for d in Path("/content").iterdir():
                if d.is_dir() and (d / "app" / "main.py").is_file() and d.name != PROJ.name:
                    d.rename(PROJ)
                    print("• تمت إعادة التسمية إلى:", PROJ.name)
                    break
    else:
        print("⬆️ لا يوجد ملف مضغوط — صعّد rafeeq-main.zip الآن (ستظهر نافذة)…")
        try:
            from google.colab import files

            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("لم يُرفع أي ملف")
            zip_name = list(uploaded)[0]
            with zipfile.ZipFile(zip_name) as zf:
                zf.extractall("/content")
            os.remove(zip_name)
        except Exception as upload_err:
            print("لم يتم الرفع — أجرب النسخة العامة من GitHub…", upload_err)
            clone = subprocess.run(
                ["git", "clone", "-q", "https://github.com/911houssem/rafeeq", str(PROJ)]
            )
            if clone.returncode != 0:
                raise RuntimeError(
                    "تعذّر جلب المشروع — ارفع rafeeq-main.zip يدويًا عبر شرائح «Files»"
                )

assert (PROJ / "app" / "main.py").is_file(), "تعذُّر الحصول على كود المشروع"
print("✅ المشروع جاهز:", PROJ)
os.chdir(PROJ)


In [ ]:
%%bash
export PIP_NO_CACHE_DIR=1 MPLBACKEND=Agg DEBIAN_FRONTEND=noninteractive
cd /content/rafeeq-main

# 1) مكتبة النظام لقراءة ملفات الصوت
(apt-get -qq update >/dev/null && apt-get -qq install -y libsndfile1 >/dev/null) || sudo apt-get -qq install -y libsndfile1

# 2) onnxruntime: جرّب نسخت GPU — وإن تعثّر استخدم cop
pip install -q onnxruntime-gpu 2>/dev/null || pip install -q onnxruntime

# 3) التبعيات الرئيسة (بعد حذف onnxruntime من الملف)
grep -v -e '^onnxruntime' deploy/requirements-server.txt > /tmp/rafeeq-req.txt
pip install -q -r /tmp/rafeeq-req.txt \
  || ( grep -v -e '^catt_tashkeel' /tmp/rafeeq-req.txt > /tmp/rafeeq-req2.txt
       pip install -q -r /tmp/rafeeq-req2.txt )

# 4) حزمة SILMA TTS بلا تبعياتها الأصلية (بديلها مجلد vendor/nemo_text_processing في كود رفيق)
pip install -q --no-deps silma-tts

# 5) فحص جاهزية الاستيرادات — يصطاد أي خطأ الآن بدلاً من لحظة تشغيل النماذج
python - <<'PY'
import sys, traceback
sys.path.insert(0, "")
FAIL = False

def go(label, fn):
    global FAIL
    try:
        fn()
        print("✅", label)
    except Exception as e:
        FAIL = True
        print("❌", label, "->", type(e).__name__, ":", e)
        traceback.print_exc(limit=1)

def basic():
    import fastapi, librosa, soundfile, transformers  # noqa

def appmod():
    import app.main  # noqa

def ttsimport():
    # نفس ما يفعله tts.py: يضيف مجلد vendor قبل استيراد silma_tts
    import sys
    from pathlib import Path

    vend = Path("/content/rafeeq-main/vendor")
    if str(vend) not in sys.path:
        sys.path.insert(0, str(vend))
    from silma_tts.infer import utils_infer  # noqa

def normalizer():
    import sys
    from pathlib import Path

    vend = Path("/content/rafeeq-main/vendor")
    if str(vend) not in sys.path:
        sys.path.insert(0, str(vend))
    from silma_tts.infer.utils_infer import load_nemo_text_normalizer  # noqa

def tashkeel():
    import sys
    from pathlib import Path

    vend = Path("/content/rafeeq-main/vendor")
    if str(vend) not in sys.path:
        sys.path.insert(0, str(vend))
    from silma_tts.infer.utils_infer import load_tashkeel_model
    try:
        load_tashkeel_model()
    except Exception as e:
        print("   (تنبيه اختياري) نموذج التشكيل غير واجب:", type(e).__name__, e)

go("المكتبات الأساسية (fastapi/librosa/soundfile/transformers)", basic)
go("import app.main — كل المسارات والمحركات", appmod)
go("silma_tts.infer.utils_infer", ttsimport)
go("nemo_text_processing (vendor)", normalizer)
go("نموذج التشكيل catt_tashkeel", tashkeel)
if FAIL:
    print("⚠️ في أخطاء استيراد أعلاه — سجل النتائج أعلاه، النظام لن يعمل صوتيًا كاملاً.")
sys.exit(1 if FAIL else 0)
PY


In [ ]:
# خلية 4 — تنزيل النماذج الثلاثة الأصلية من HuggingFace (بنفس أسماء ملفاتك حرفيًا)
# - تدعم الإعادة: لو انقطع، أعد تنفيذ الخلية وستُستأنف من حيث توقفت.
# - يبحث أولًا عن نسخة محفوظة في Drive في المجلد /content/drive/MyDrive/rafeeq_models
import os
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path

PROJ = Path("/content/rafeeq-main")
ROOT = Path("/content/models")
DRIVE_MODELS = Path("/content/drive/MyDrive/rafeeq_models")


def complete_models(root):
    '''تحقق: asr/llm يحتويان config.json، و tts يحتوي model.pt.'''
    if not root.is_dir():
        return False
    return (
        (root / "asr" / "config.json").is_file()
        and (root / "llm" / "config.json").is_file()
        and (root / "tts" / "model.pt").is_file()
    )


def drive_mounted():
    return Path("/content/drive/MyDrive").exists()


if complete_models(ROOT):
    print("✅ النماذج موجودة محلّيًا — نتجاوز التنزيل.")
elif drive_mounted() and complete_models(DRIVE_MODELS):
    print("🔁 نسخ النماذج من Google Drive… (5–10 دقائق من درايف)")
    for d in ("asr", "llm", "tts"):
        dest = ROOT / d
        if not dest.exists():
            shutil.copytree(DRIVE_MODELS / d, dest)
    print("✅ تم الاسترجاع من Drive.")
else:
    hf_token = os.environ.get("HF_TOKEN", "").strip()
    try:
        entered = getpass("اختياري: عند طلب توثيق من HuggingFace أدخل HF_TOKEN (اضغط Enter للتجاوز): ").strip()
        hf_token = hf_token or entered
    except Exception:
        pass
    env = {**os.environ, "RAFEEQ_MODELS_ROOT": str(ROOT)}
    if hf_token:
        env["HF_TOKEN"] = hf_token
    print("⬇️ بدء تنزيل النماذج (~33GB — بين 5 و20 دقيقة حسب سرعة الكولاب)")
    r = subprocess.run([sys.executable, "deploy/download_models.py"], cwd=PROJ, env=env)
    if r.returncode != 0 or not complete_models(ROOT):
        raise RuntimeError(
            "فشل التنزيل (غالبًا انقطاع) — أعد تنفيذ هذه الخلية: سيُستأنف من حيث توقف"
        )

# الأرقام النهائية
import shutil as _sh

sizes = {
    k: sum(f.stat().st_size for f in (ROOT / k).rglob("*") if f.is_file()) / 1e9
    for k in ("asr", "llm", "tts")
}
for _k, _s in sizes.items():
    print(f"  • {_k}: {_s:.1f} GB")
print(f"  ↳ المساحة الفارغة على القرص: {_sh.disk_usage('/content').free / 1e9:.0f} GB")


In [ ]:
# خلية 5 — (اختياري) حفظ النماذج في Google Drive لجعل الجلسات القادمة أسرع.
# ⚠️ تحتاج ~33GB حرة في Drive. بعد الحفظ، الخلية 4 تأخذ نسخة من هنا تلقائيًا في كل جلسة جديدة.
SAVE_TO_DRIVE = False  # ← اضبطها True ثم شغّل الخلية

from pathlib import Path
ROOT = Path("/content/models")
DRIVE_MODELS = Path("/content/drive/MyDrive/rafeeq_models")

if not SAVE_TO_DRIVE:
    print("(متجاهل) — لتسريع الجلسات القادمة: اضبط True ثم شغّل الخلية مرة أخرى.")
else:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    import shutil

    print("⏳ نسخ النماذج إلى Drive — يستغرق عشرات الدقائق أول مرة فقط…")
    shutil.copytree(str(ROOT), str(DRIVE_MODELS), dirs_exist_ok=True)
    print("✅ تم الحفظ في:", DRIVE_MODELS, "— الجلسات القادمة ستستنسخ من هنا وتتخطّى مرحلة التنزيل.")


In [ ]:
# خلية 6 — تشغيل خادم رفيق في الخلفية على المنفذ 8000
import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import torch

PROJ = Path("/content/rafeeq-main")
ROOT = Path("/content/models")
PORT = 8000

# أوقف أي خادم قديم
subprocess.run(["pkill", "-f", "uvicorn app.main:app"], capture_output=True)

env = {
    **os.environ,
    "RAFEEQ_HOST": "0.0.0.0",
    "RAFEEQ_PORT": str(PORT),
    "RAFEEQ_MODEL_ASR_DIR": str(ROOT / "asr"),
    "RAFEEQ_MODEL_LLM_DIR": str(ROOT / "llm"),
    "RAFEEQ_MODEL_TTS_DIR": str(ROOT / "tts"),
    "RAFEEQ_DATA_DIR": str(PROJ / "data"),
    "PYTHONIOENCODING": "utf-8",
    "MPLBACKEND": "Agg",
    "PIP_NO_CACHE_DIR": "1",
}

# كارت T4 لا يدعم bfloat16 بكفاءة — نضبط float16 تلقائيًا
_gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
if "T4" in _gpu:
    env["RAFEEQ_LLM_DTYPE"] = "float16"
    env["RAFEEQ_ASR_DTYPE"] = "float16"
    print("ℹ️ T4: تم ضبط الدقة float16 تلقائيًا.")

log_path = PROJ / "server.log"
if log_path.exists():
    log_path.write_bytes(b"")  # مسح سجل قديم

log = open(log_path, "ab")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=str(PROJ),
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)


def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=3) as r:
            return json.loads(r.read())
    except Exception:
        return None


_up = None
for _ in range(90):  # حتى 3 دقائق لانتظار الإقلاع
    time.sleep(2)
    _up = health()
    if _up:
        break

if _up:
    print("✅ الخادم يعمل —", json.dumps(_up, ensure_ascii=False))
    # انتظر حتى تكتمل النماذج الثلاثة (حتى ~12 دقيقة) مع إظهار الحال حيًا
    import urllib.request as _u

    _ready = False
    for _i in range(36):
        time.sleep(20)
        try:
            with _u.urlopen(f"http://127.0.0.1:{PORT}/api/status", timeout=5) as _r:
                _st = json.loads(_r.read())
        except Exception:
            continue
        _line = " | ".join(f"{k}:{v.get('status')}" for k, v in (_st.get("models") or {}).items())
        print(f"  ⏳ تحميل… ({_i + 1}) {_line}")
        if _st.get("ready"):
            _ready = True
            print("🎉 كل النماذج جاهزة — افتح الرابط من الخلية 7 الآن")
            break
    if not _ready:
        print("⏰ بعض النماذج لسه بتُحمَّل أو واجهت خطأ — راجع خلية 8 للحالة التفصيلية (يمكنك المتابعة، النفق يعمل)")
else:
    print("❌ لم يستجب الخادم بعد — أحدث أسطر السجل:")
    print(log_path.read_text(encoding="utf-8", errors="ignore")[-1500:])
    raise RuntimeError("افحص السجل أعلاه وأعد تشغيل هذه الخلية")


In [ ]:
# خلية 7 — فتح رابط عمومي مجاني فوري (نفق cloudflared — يدعم WebSocket للصوت)
import os
import re
import subprocess
import time
from pathlib import Path

BIN = "/usr/local/bin/cloudflared"
tlog = Path("/content/tunnel.log")

# أوقف النفق القديم إن وُجد
subprocess.run(["pkill", "-f", "cloudflared tunnel"], capture_output=True)

if not os.path.exists(BIN):
    print("⬇️ تحميل cloudflared…")
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", BIN,
        ],
        check=True,
    )
    os.chmod(BIN, 0o755)

if tlog.exists():
    tlog.write_bytes(b"")

lg = open(tlog, "wb")
subprocess.Popen(
    [BIN, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=lg,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

url = None
print("⏳ فتح النفق…")
for _ in range(30):
    time.sleep(2)
    try:
        txt = tlog.read_text(encoding="utf-8", errors="ignore")
    except FileNotFoundError:
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        url = m.group(0)
        break

if url:
    print("🔗 رابط رفيق العمومي:", url)
    print("1. افتح الرابط من متصفحك.")
    print("2. أنشئ حساب المالك (يُغلق مباشرة بعد أول إنشاء).")
    print("3. مهم: لا تُغلق الكولاب! لو ضاع الرابط، أعد تشغيل هذه الخلية فقط.")
else:
    print("❌ تعذّر الحصول على الرابط — آخر أسطر السجل:")
    print(tlog.read_text(encoding="utf-8", errors="ignore")[-800:])


In [ ]:
# خلية 8 — الحالة المباشرة: النماذج + سجل الخادم (نفّذها عند أي مشكلة)
import json
import urllib.request
from pathlib import Path


def _get(path):
    with urllib.request.urlopen(f"http://127.0.0.1:8000{path}", timeout=5) as r:
        return json.loads(r.read())


try:
    st = _get("/api/status")
    print("👑 حساب المالك:", "موجود" if st.get("owner_exists") else "غير منشأ بعد (افتح الرابط وأنشئه)")
    for _k, _m in (st.get("models") or {}).items():
        print(f"  • {_k}: {_m.get('status')}")
    print("✅ كل النماذج جاهزة" if st.get("ready") else "⏳ النماذج لسه بتُحمَّل…")
except Exception as e:
    print("❌ الخادم غير مستجيب:", e)

print("\n--- آخر أسطر server.log ---")
print(Path("/content/rafeeq-main/server.log").read_text(encoding="utf-8", errors="ignore")[-2500:])


## 🧪 اختبار API (اختياري)
نفّذ بعد إنشاء حساب المالك من الرابط العمومي. التجربة تعيد جوابًا نصيًا + ملف WAV بصوتك. المالك لا يُفوتر، لكن أي حساب عادي يُخصم من دقائقه.


In [ ]:
# خلية 9 — تجربة محادثة نصية + نطق عبر واجهة متوافقة مع OpenAI
from getpass import getpass

import requests

BASE = "http://127.0.0.1:8000"
MODEL = "jais-3-7b"

email = input("بريد حساب المالك: ").strip()
pw = getpass("كلمة المرور: ")

r = requests.post(f"{BASE}/api/auth/login", json={"email": email, "password": pw})
r.raise_for_status()
token = r.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}
print("✅ تم الدخول")

# ① محادثة نصية
r2 = requests.post(
    f"{BASE}/v1/chat/completions",
    headers=headers,
    json={"model": MODEL, "messages": [{"role": "user", "content": "عرف نفسك بكلمتين"}]},
    timeout=300,
)
r2.raise_for_status()
reply = r2.json()["choices"][0]["message"]["content"]
print("🤝 جواب رفيق:", reply)

# ② نطق الجواب (SILMA TTS)
r3 = requests.post(
    f"{BASE}/v1/audio/speech",
    headers=headers,
    json={"model": "silma-tts", "input": reply},
    timeout=300,
)
r3.raise_for_status()
with open("/content/rafeeq_reply.wav", "wb") as f:
    f.write(r3.content)
print("🔊 الملف: /content/rafeeq_reply.wav — اضغط للتشغيل:")
from IPython.display import Audio, display

display(Audio("/content/rafeeq_reply.wav"))


## 📌 بعد التشغيل — قواعد الجلسة
- **الحسابات والفوترة** داخل قاعدة SQLite محلية (`data/app.db`) — كل جلسة كولاب تعاد من الصفر. تريد الاحتفاظ بها؟ اضبط `RAFEEQ_DATA_DIR` على مجلد Drive الخاص بك (خلية 6).
- كولاب **يقطع الجلسة** بعد فترة خمول (عادة 10–90 دقيقة، وبحد أعلى 12 ساعة) — بعد القطع تفقد كل شيء من النماذج والرابط.
- الرابط **عمومي للعالم**: لا تشارك كلمة مرور المالك، وأغلق `allow_register` من لوحة الإدارة بعد تجريبك.
- **للإنتاج الحقيقي**: كولاب غير مناسب (قطع 12 ساعة + بلا استمرارية) — استخدم RunPod أو Modal كما في `deploy/README.md` (يعمل 24/7 بنفس النماذج بدون أي تعديل).
- 🆘 إن ارتفع الذاكرة فوق حدّ A100: خفّض `RAFEEQ_LLM_DTYPE=float16`، أو شغّل النماذج واحدًا تلو الآخر من لوحة المالك (`warm_models=0`).
- 🧩 الميكروفون يحتاج HTTPS — النفق يوفر HTTPS، جرّبه من متصفح حديث وامنح صلاحية الميكروفون.

## 🧭 تجربة الصوت الحيّ
في الواجهة: زر 🎤 الميكروفون — انطق جملة عربية وانتظر (من 3 إلى 20 ثانية حسب الكارت)، وبعدها تسمع رد بالصوت. المقاطعة (barge-in) تعمل تلقائيًا: ابدأ بالحديث أثناء نطق المتصفح وسيتوقف ويعيد التفكير.
